## 1. Fetch Statistics from Source NetCDF (Arctic Rivers)

Replaces the Rasdaman WCS approach. Reads `mhit_indices_combined_for_rasdaman.nc` — the source
file used to populate the Rasdaman coverage — and computes ensemble delta statistics, writing
`arctic_stats.csv` with the same 66-column structure as the CONUS `stats_diff.csv`.

**Source dims:** `(stream_id: 34346, source: 3, era: 2, model: 7)`
- model: `{0:C2LE2, 1:C2LE4, 2:C2LE7, 3:C2LE9, 4:PGWh, 5:PGWm, 6:historical}`
- source: `{0:original_gcm, 1:gcm_diff, 2:gcm_diff_applied_to_blaskey}`
- era: `{0:1990-2021, 1:2034-2065}`

**Methodology:**
- C2LE models (0-3): `source=2` (gcm_diff_applied_to_blaskey), `era=1` — Blaskey ratio pre-applied
- PGW models (4-5): `source=0` (original_gcm), `era=1` — no historical run, ratio cannot be applied
- Historical baseline: `source=0`, `model=6`, `era=0`
- Ensemble min/avg/max computed across all 6 projected models combined

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

In [2]:
NC_PATH  = Path("/beegfs/CMIP6/crstephenson/arctic_rivers_data/mhit_indices_combined_for_rasdaman.nc")
CSV_PATH = Path("arctic_stats.csv")

# Dimension integer encodings — int64 coords in this file
SOURCE_BLASKEY = 2   # gcm_diff_applied_to_blaskey (C2LE projected)
SOURCE_ORIG  = 0   # original_gcm (PGW projected + historical baseline)
ERA_PROJ     = 1   # 2034-2065
ERA_HIST     = 0   # 1990-2021
MODEL_HIST   = 6   # historical (Blaskey baseline)

C2LE_MODELS = [0, 1, 2, 3]   # C2LE2, C2LE4, C2LE7, C2LE9
PGW_MODELS  = [4, 5]         # PGWh, PGWm

In [3]:
VARS_META = {
    "ma12": {"stats": ["ma12_min_d", "ma12_avg_d", "ma12_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma13": {"stats": ["ma13_min_d", "ma13_avg_d", "ma13_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma14": {"stats": ["ma14_min_d", "ma14_avg_d", "ma14_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma15": {"stats": ["ma15_min_d", "ma15_avg_d", "ma15_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma16": {"stats": ["ma16_min_d", "ma16_avg_d", "ma16_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma17": {"stats": ["ma17_min_d", "ma17_avg_d", "ma17_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma18": {"stats": ["ma18_min_d", "ma18_avg_d", "ma18_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma19": {"stats": ["ma19_min_d", "ma19_avg_d", "ma19_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma20": {"stats": ["ma20_min_d", "ma20_avg_d", "ma20_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma21": {"stats": ["ma21_min_d", "ma21_avg_d", "ma21_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma22": {"stats": ["ma22_min_d", "ma22_avg_d", "ma22_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "ma23": {"stats": ["ma23_min_d", "ma23_avg_d", "ma23_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
    "dh1":  {"stats": ["dh1_min_d",  "dh1_max_d"],                 "aggs": ["min","max"],       "diff": "pct"},
    "dl1":  {"stats": ["dl1_min_d",  "dl1_max_d"],                 "aggs": ["min","max"],       "diff": "pct"},
    "dh15": {"stats": ["dh15_min_d", "dh15_avg_d", "dh15_max_d"], "aggs": ["min","mean","max"], "diff": "abs"},
    "dl16": {"stats": ["dl16_min_d", "dl16_avg_d", "dl16_max_d"], "aggs": ["min","mean","max"], "diff": "abs"},
    "fh1":  {"stats": ["fh1_min_d",  "fh1_avg_d",  "fh1_max_d"],  "aggs": ["min","mean","max"], "diff": "abs"},
    "fl1":  {"stats": ["fl1_min_d",  "fl1_avg_d",  "fl1_max_d"],  "aggs": ["min","mean","max"], "diff": "abs"},
    "th1":  {"stats": ["th1_min_d",  "th1_avg_d",  "th1_max_d"],  "aggs": ["min","mean","max"], "diff": "circular"},
    "tl1":  {"stats": ["tl1_min_d",  "tl1_avg_d",  "tl1_max_d"],  "aggs": ["min","mean","max"], "diff": "circular"},
    "ma99": {"stats": ["ma99_min_d", "ma99_avg_d", "ma99_max_d"], "aggs": ["min","mean","max"], "diff": "pct"},
}

CSV_COLUMNS = (
    [s for v in ["ma12","ma13","ma14","ma15","ma16","ma17","ma18","ma19","ma20","ma21","ma22","ma23"]
     for s in VARS_META[v]["stats"]]
    + ["dh1_min_d","dh1_max_d","dl1_min_d","dl1_max_d"]
    + [s for v in ["dh15","dl16","fh1","fl1","th1","tl1"] for s in VARS_META[v]["stats"]]
    + ["ma99_min_d","ma99_avg_d","ma99_max_d","ma99_hist",
       "dh15_hist","dl16_hist","fh1_hist","fl1_hist"]
)
print(f"CSV_COLUMNS: {len(CSV_COLUMNS)} columns")

CSV_COLUMNS: 66 columns


In [4]:
ds = xr.open_dataset(NC_PATH)
print(f"Loaded {NC_PATH.name}: {dict(ds.sizes)}")

# C2LE projected: bias-corrected to Blaskey baseline
c2le = ds.sel(source=SOURCE_BLASKEY, era=ERA_PROJ, model=C2LE_MODELS).load()

# PGW projected: original GCM (no historical era, so gcm_diff_applied_to_blaskey unavailable)
pgw  = ds.sel(source=SOURCE_ORIG,  era=ERA_PROJ, model=PGW_MODELS).load()

# Blaskey historical baseline
hist = ds.sel(source=SOURCE_ORIG,  era=ERA_HIST, model=MODEL_HIST).load()

# Combine all 6 projected models along model dim
proj = xr.concat([c2le, pgw], dim="model")

print(f"Projected: {dict(proj.sizes)}  (C2LE x4 + PGW x2)")
print(f"Historical: {dict(hist.sizes)}")

Loaded mhit_indices_combined_for_rasdaman.nc: {'source': 3, 'era': 2, 'model': 7, 'stream_id': 34346}
Projected: {'model': 6, 'stream_id': 34346}  (C2LE x4 + PGW x2)
Historical: {'stream_id': 34346}


/beegfs/CMIP6/crstephenson/tmp/ipykernel_1247894/3743607565.py:14: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  proj = xr.concat([c2le, pgw], dim="model")


In [5]:
rows = {}
for var, meta in VARS_META.items():
    p = proj[var]   # (model, stream_id)
    h = hist[var]   # (stream_id,)
    if meta["diff"] == "pct":
        denom  = xr.where(h != 0, h, 0.0001)
        deltas = (p - h) / denom * 100
    elif meta["diff"] == "circular":
        deltas = ((p - h + 183) % 366) - 183
    else:
        deltas = p - h
    for stat, agg in zip(meta["stats"], meta["aggs"]):
        rows[stat] = getattr(deltas, agg)(dim="model").round(2).values

for v in ("ma99", "dh15", "dl16", "fh1", "fl1"):
    rows[f"{v}_hist"] = hist[v].round(2).values

df = pd.DataFrame(rows, index=hist.stream_id.values)
df.index.name = "stream_id"
df = df[CSV_COLUMNS]
print(f"Computed: {len(df)} rows x {len(df.columns)} columns")

Computed: 34346 rows x 66 columns


In [6]:
df.to_csv(CSV_PATH)
print(f"Wrote {CSV_PATH}")

Wrote arctic_stats.csv


In [7]:
df_check = pd.read_csv(CSV_PATH, index_col="stream_id")
print(f"arctic_stats.csv: {len(df_check)} rows, {len(df_check.columns)} columns")
print(f"NaN rows: {df_check.isna().all(axis=1).sum()}")
df_check.head(2)

arctic_stats.csv: 34346 rows, 66 columns
NaN rows: 0


,ma12_min_d,ma12_avg_d,ma12_max_d,ma13_min_d,ma13_avg_d,ma13_max_d,ma14_min_d,ma14_avg_d,ma14_max_d,ma15_min_d,...,tl1_avg_d,tl1_max_d,ma99_min_d,ma99_avg_d,ma99_max_d,ma99_hist,dh15_hist,dl16_hist,fh1_hist,fl1_hist
stream_id,,,,,,,,,,,,,,,,,,,,,
81000004,-44.30,25.49,248.38,-41.19,-6.44,76.50,-36.32,-15.55,18.49,-0.52,...,-5.61,-3.04,3.67,9.95,19.05,4160.15,22.75,74.0,3.68,0.97
81000005,-31.24,40.59,216.36,-63.35,-20.53,33.89,-39.89,-7.65,45.28,1.95,...,-6.47,-2.41,1.60,8.32,16.69,3404.80,32.67,87.0,3.19,0.87
